# Two logical qubits → Bell state via logical $H$ + logical CNOT

This notebook simulates two $d=5$ rotated surface-code patches on a single
MPS, walks through the standard Bell-state preparation circuit at the
logical level, and reads out the resulting two-qubit state.

## Protocol
1. Prepare $|0\rangle_L \otimes |0\rangle_L$ — every data qubit in $|0\rangle$.
2. Run several rounds of Z-stabilizer measurements on both patches to project
   each into a definite-syndrome representative of $|0\rangle_L$ and provide
   a "before" detector pattern.
3. Apply **transversal Hadamard** on patch 1: $H$ on every data qubit of
   patch 1.
4. Run more rounds of Z-stabilizer measurements. Patch 2 is undisturbed by
   step 3, so its Z-syndromes still detect physical X errors. Patch 1's
   stabilizer labels are conceptually *swapped* by the transversal H, but the
   Pauli frame and decoder logic still work on the same hardware — we just
   note the relabel in the markdown.
5. Apply **transversal CNOT** between corresponding data qubits of the two
   patches.
6. Run more rounds of Z-stabilizer measurements.
7. Destructively measure every data qubit in the Z-basis. Compute $Z_{L,1}$
   and $Z_{L,2}$ (each with their Pauli frames). Repeat the whole protocol
   many times — the outcome distribution should be:
   $(0,0)$ and $(1,1)$ each $\approx 50\%$, $(0,1)$ and $(1,0)$ vanishingly
   rare. That's the signature of $|\Phi^+\rangle_L = (|00\rangle_L + |11\rangle_L)/\sqrt{2}$.

## 1 · Imports + bond-dimension cap

`threshold = 1e-8` everywhere — looser than the `1e-12` of the earlier
notebooks — so the MPS bond dimension can't blow up under the two-patch
load.

In [1]:
using ITensors, ITensorMPS, LinearAlgebra, Plots, Random
Random.seed!(0xB011)
const threshold = 1e-6     # MPS truncation cutoff
gr();

## 2 · Two-patch lattice

Each patch is the same staggered $d=5$ rotated surface code as in the
previous notebooks. We give each data and ancilla coordinate a *patch
label* `p ∈ {1, 2}` so that `(p, (x, y))` uniquely identifies a qubit
position.

**MPS ordering is side-by-side: all of patch 1, then all of patch 2.**
Within each patch we go data → Z-aux → X-aux in the natural order. This is
deliberately the *non-locality-revealing* ordering: the transversal CNOT
between patch 1's data qubit $i$ and patch 2's data qubit $i$ now spans
roughly half the MPS, which means the truncation cutoff `threshold = 1e-8`
gets to do real work. The point of running this experiment is to see how
that non-local gate behaves so we know what to expect when scaling to more
patches.

In [2]:
# d = 3 keeps the transversal CNOT (which spans ~half the MPS in our
# side-by-side layout) tractable. Crank up to d = 5 once you've validated
# the rest of the pipeline and are willing to budget the runtime.
const d = 3

# Geometry for a single patch (same as the no-reset notebook)
data_coords = vec([(x, y) for x in 0:(d-1), y in 0:(d-1)])
z_aux_local = Tuple{Float64,Float64}[]
x_aux_local = Tuple{Float64,Float64}[]
for x in 0:(d-2), y in 0:(d-2)
    push!((x + y) % 2 == 0 ? z_aux_local : x_aux_local, (x + 0.5, y + 0.5))
end
for y in 1:2:(d-2);  push!(z_aux_local, (-0.5,    y + 0.5)); end
for y in 0:2:(d-2);  push!(z_aux_local, (d - 0.5, y + 0.5)); end
for x in 0:2:(d-2);  push!(x_aux_local, (x + 0.5, -0.5));    end
for x in 1:2:(d-2);  push!(x_aux_local, (x + 0.5, d - 0.5)); end

const Nz_per_patch = length(z_aux_local)  # 12 for d=5

# MPS ordering: side-by-side. All of patch 1 first, then all of patch 2.
# Within each patch: data, then Z-aux, then X-aux.
# Trade-off: keeps the within-patch CNOT chains short, but the transversal
# CNOT between the two patches now spans roughly half the chain — exactly
# the kind of non-local 2-qubit gate we want to stress-test.
all_keys = Tuple{Int, Tuple{Float64,Float64}}[]
for p in 1:2
    for q in data_coords;  push!(all_keys, (p, q)); end
    for a in z_aux_local;  push!(all_keys, (p, a)); end
    for a in x_aux_local;  push!(all_keys, (p, a)); end
end

sites = siteinds("S=1/2", length(all_keys))
site_of = Dict{Tuple{Int, Tuple{Float64,Float64}}, Index}(
    k => sites[i] for (i, k) in enumerate(all_keys))

# Bookkeeping for diagnostics: index of each site in the MPS
mps_index_of = Dict(k => i for (i, k) in enumerate(all_keys))

println("total sites: ", length(all_keys),
        "  (",  2 * length(data_coords), " data + ",
        2 * length(z_aux_local), " Z-aux + ",
        2 * length(x_aux_local), " X-aux)")
println("transversal CNOT separation: data(p=1,(0,0)) is site ",
        mps_index_of[(1, (0, 0))],
        ", data(p=2,(0,0)) is site ",
        mps_index_of[(2, (0, 0))],
        "  — gap ", mps_index_of[(2, (0, 0))] - mps_index_of[(1, (0, 0))])

total sites: 34  (18 data + 8 Z-aux + 8 X-aux)
transversal CNOT separation: data(p=1,(0,0)) is site 1, data(p=2,(0,0)) is site 18  — gap 17


## 3 · Per-qubit / per-patch helpers

In [3]:
# Single-patch data-neighbor lookup
data_neighbors_of(aux_coord) =
    [d for d in data_coords if abs(d[1]-aux_coord[1])==0.5 && abs(d[2]-aux_coord[2])==0.5]

P_up(s) = 0.5 * op("Id", s) + op("Sz", s)
P_dn(s) = 0.5 * op("Id", s) - op("Sz", s)

# Z-basis projective measurement: bit=0 ⇔ Up, bit=1 ⇔ Dn
function measure_Z!(psi, site; cutoff = threshold)
    sz = real(inner(psi', apply(op("Sz", site), psi; cutoff)))
    if rand() < 0.5 + sz
        psi = apply(P_up(site), psi; cutoff); bit = 0
    else
        psi = apply(P_dn(site), psi; cutoff); bit = 1
    end
    bit, psi / sqrt(real(inner(psi, psi)))
end

# One Z-stab measurement on patch p at the given aux coord (no-reset model)
function measure_Z_stab(psi, p::Int, aux_coord; cutoff = threshold)
    aux_site = site_of[(p, aux_coord)]
    nbrs = data_neighbors_of(aux_coord)
    order = length(nbrs) == 4 ? [2, 4, 1, 3] : [1, 2]
    for q in nbrs[order]
        psi = apply(op("CNOT", site_of[(p, q)], aux_site), psi; cutoff)
    end
    measure_Z!(psi, aux_site; cutoff)
end;

## 4 · X-stabilizer measurement and code-space projection

The previous notebooks only measured Z-stabilizers, which is enough to decode
X errors when the state stays inside the codespace. Here we also need X-stab
measurements at the start, for one specific reason:

> $|0\rangle^{\otimes n}$ on the *data* is **not** $|0\rangle_L$ of the
> surface code — it's a $50/50$ superposition over all X-syndrome sectors.
> Without X-stab measurements, the post-CNOT state would be 25 *physical*
> Bell pairs and the MPS bond dimension across the patch cut would be
> $2^{25}$. Measuring both stabilizer types projects each patch into a
> 2-dimensional code space, so the joint state lives in a $4$-d logical
> space and the bond dimension stays bounded.

X-stab circuit: $H$ on aux, $\text{CNOT}_{\mathrm{aux}\to\mathrm{data}}$ from
the auxiliary to every data qubit in the support, then $H$ on aux, then
measure in Z (see Fig. 2(a) of the paper).

In [4]:
# Reset an ancilla to Up (= |0>). Used as a one-shot housekeeping step at
# the start, and between stabilizer-type changes.
function reset_aux!(psi, aux_site; cutoff = threshold)
    sz = real(inner(psi', apply(op("Sz", aux_site), psi; cutoff)))
    if sz < 0
        psi = apply(op("X", aux_site), psi; cutoff)
    end
    psi
end

# One-shot X-stab measurement: H — CNOTs (aux→data) — H — Z-measure aux.
# Assumes the aux qubit has been reset to |0> already.
function measure_X_stab(psi, p::Int, aux_coord; cutoff = threshold)
    aux_site = site_of[(p, aux_coord)]
    psi = apply(op("H", aux_site), psi; cutoff)
    nbrs = data_neighbors_of(aux_coord)
    order = length(nbrs) == 4 ? [2, 1, 4, 3] : [1, 2]
    for q in nbrs[order]
        psi = apply(op("CNOT", aux_site, site_of[(p, q)]), psi; cutoff)
    end
    psi = apply(op("H", aux_site), psi; cutoff)
    measure_Z!(psi, aux_site; cutoff)
end

# Project a patch into its codespace once. Measure every Z-stab and every
# X-stab (each ancilla reset beforehand). Returns the two syndrome vectors.
function project_to_codespace!(psi, p::Int; cutoff = threshold)
    z_syn = Int[]
    for ac in z_aux_local
        psi = reset_aux!(psi, site_of[(p, ac)]; cutoff)
        bit, psi = measure_Z_stab(psi, p, ac; cutoff)
        push!(z_syn, bit)
    end
    x_syn = Int[]
    for ac in x_aux_local
        psi = reset_aux!(psi, site_of[(p, ac)]; cutoff)
        bit, psi = measure_X_stab(psi, p, ac; cutoff)
        push!(x_syn, bit)
    end
    (; z_syn, x_syn), psi
end;

## 5 · Multi-round Z-syndrome readout (reset model)

After the initial codespace projection, we reset each ancilla and run
$R$ rounds of Z-stabilizer measurements, recording the raw bit each round.
This is the data the decoder will work on for detecting X errors.

In [5]:
# Run R rounds of Z-stab measurements on both patches, with reset between
# rounds. Optionally inject physical X errors (Dict mapping round -> list of
# (patch, data_coord)) and measurement errors (Dict mapping round ->
# list of (patch, stab_idx)).
#
# Returns:
#   raw[p][r]  = vector of Nz_per_patch bits, the recorded Z-stab outcomes
#                of patch p at round r.
function run_Z_rounds!(psi, R::Int;
        physical_X = Dict{Int, Vector{Tuple{Int, Tuple{Int,Int}}}}(),
        meas_err   = Dict{Int, Vector{Tuple{Int, Int}}}(),
        cutoff     = threshold)
    raw = Dict(1 => [Int[] for _ in 1:R], 2 => [Int[] for _ in 1:R])
    for r in 1:R
        # Physical X errors injected *before* the round-r measurements
        for (p, q) in get(physical_X, r, Tuple{Int, Tuple{Int,Int}}[])
            psi = apply(op("X", site_of[(p, q)]), psi; cutoff)
        end

        for p in 1:2, (i, ac) in enumerate(z_aux_local)
            psi = reset_aux!(psi, site_of[(p, ac)]; cutoff)
            bit, psi = measure_Z_stab(psi, p, ac; cutoff)
            push!(raw[p][r], bit)
        end

        # Apply classical measurement-error flips to the recorded bits
        for (p, i) in get(meas_err, r, Tuple{Int, Int}[])
            raw[p][r][i] = 1 - raw[p][r][i]
        end
    end
    raw, psi
end;

## 6 · Logical $H$ and transversal CNOT

- **Logical $H$**: apply Hadamard to every data qubit of patch 1. On
  $|0\rangle_L$ this gives $|+\rangle_L$. It also *swaps* the X and Z
  stabilizer roles of patch 1 — what we still call "Z-stabs" after H are
  measuring what used to be the X-stabs. We'll deliberately *not* measure
  Z-stabs immediately after $H$; the next syndrome round comes after CNOT,
  so the relabel is invisible.
- **Transversal CNOT**: apply $\text{CNOT}(d^{(1)}_i, d^{(2)}_i)$ between
  every pair of corresponding data qubits. With our side-by-side MPS
  ordering each of these spans $\sim 49$ sites, so this step is the
  bond-dimension stress test.

In [6]:
# Apply transversal H to patch p's data qubits.
function logical_H!(psi, p::Int; cutoff = threshold)
    for q in data_coords
        psi = apply(op("H", site_of[(p, q)]), psi; cutoff)
    end
    psi
end

# Apply transversal CNOT(p_ctrl, p_tgt). Each gate spans roughly half the MPS.
function logical_CNOT!(psi, p_ctrl::Int, p_tgt::Int; cutoff = threshold)
    for q in data_coords
        psi = apply(op("CNOT", site_of[(p_ctrl, q)], site_of[(p_tgt, q)]), psi; cutoff)
    end
    psi
end;

## 7 · Per-patch matching graph (reset model, 3-D)

Same matching-graph structure as `example_decoder_3d.ipynb`, but built once
per patch. We use the **reset model** here (each ancilla reset between
rounds, so time edges go $r \leftrightarrow r{+}1$).

Each patch's graph is built independently of the other; decoding is also
per patch. After the CNOT, a single physical X on patch 1 propagates to a
correlated X on patch 2 — independent decoders see that as two separate
errors and each lands a correction in the right place, so the overall
Pauli frame still tracks the true accumulated error correctly.

In [7]:
# Spatial matching-graph bookkeeping for ONE patch (geometry shared by both).
z_aux_containing(q) =
    [i for (i, a) in enumerate(z_aux_local)
       if abs(a[1]-q[1])==0.5 && abs(a[2]-q[2])==0.5]

edge_data_local  = Dict{Tuple{Int,Int}, Tuple{Int,Int}}()
bedge_data_local = Dict{Int, Tuple{Int,Int}}()
for q in data_coords
    zs = z_aux_containing(q)
    if length(zs) == 2
        a, b = minmax(zs[1], zs[2])
        edge_data_local[(a, b)] = q
    elseif length(zs) == 1
        bedge_data_local[zs[1]] = q
    end
end

# 3-D matching graph for a given number of rounds R (reset model -> r↔r+1).
function build_graph(R::Int)
    Nz   = Nz_per_patch
    Ntot = R * Nz + 1
    BND  = Ntot
    node_id(i, r) = (r - 1) * Nz + i
    decode_id(u)  = (mod1(u, Nz), div(u - 1, Nz) + 1)

    dist = fill(Inf, Ntot, Ntot)
    nxt  = fill(-1,  Ntot, Ntot)
    for i in 1:Ntot
        dist[i, i] = 0.0; nxt[i, i] = i
    end
    add!(u, v) = begin
        dist[u, v] = 1.0; dist[v, u] = 1.0
        nxt[u, v]  = v;   nxt[v, u]  = u
    end
    for r in 1:R, ((i, j), _) in edge_data_local
        add!(node_id(i, r), node_id(j, r))
    end
    for r in 1:(R-1), i in 1:Nz
        add!(node_id(i, r), node_id(i, r + 1))
    end
    for r in 1:R, (i, _) in bedge_data_local
        add!(node_id(i, r), BND)
    end
    for k in 1:Ntot, i in 1:Ntot, j in 1:Ntot
        if dist[i, k] + dist[k, j] < dist[i, j]
            dist[i, j] = dist[i, k] + dist[k, j]
            nxt[i, j]  = nxt[i, k]
        end
    end
    function edge_kind(u, v)
        u, v = minmax(u, v)
        if v == BND
            i, _ = decode_id(u); return (:boundary, bedge_data_local[i])
        end
        iu, ru = decode_id(u); iv, rv = decode_id(v)
        if ru == rv
            a, b = minmax(iu, iv); return (:spatial, edge_data_local[(a, b)])
        else
            return (:time, (iu, min(ru, rv)))
        end
    end
    function path(a, b)
        nxt[a, b] == -1 && return Int[]
        p = [a]
        while a != b; a = nxt[a, b]; push!(p, a); end
        p
    end
    (; dist, nxt, BND, node_id, decode_id, edge_kind, path, R)
end;

## 8 · MWPM + per-patch decoder

In [8]:
function mwpm(defects::Vector{Int}, dist::Matrix{Float64}, BND::Int)
    isempty(defects) && return Tuple{Int,Int}[], 0.0
    nd    = length(defects)
    nodes = vcat(defects, fill(BND, nd))
    N     = length(nodes)
    W = zeros(N, N)
    for i in 1:N, j in (i+1):N
        W[i, j] = (nodes[i] == BND && nodes[j] == BND) ? 0.0 :
                  dist[nodes[i], nodes[j]]
        W[j, i] = W[i, j]
    end
    function rec(rem)
        isempty(rem) && return 0.0, Tuple{Int,Int}[]
        first = rem[1]; best_c = Inf; best_m = Tuple{Int,Int}[]
        for k in 2:length(rem)
            c = W[first, rem[k]]
            isfinite(c) || continue
            sub = vcat(rem[2:k-1], rem[k+1:end])
            sc, sm = rec(sub)
            if c + sc < best_c
                best_c = c + sc; best_m = vcat([(first, rem[k])], sm)
            end
        end
        best_c, best_m
    end
    cost, m_idx = rec(collect(1:N))
    pairs = [(nodes[i], nodes[j]) for (i, j) in m_idx
             if !(nodes[i] == BND && nodes[j] == BND)]
    pairs, cost
end

# Decode one patch's syndrome history into a Pauli-frame X-correction list.
# `raw_history[r]` is the round-r vector of bits for this patch.
# `initial_syn` is the round-0 syndrome (from the codespace-projection step),
# included so the first detector compares round 1 against it.
function decode_patch(raw_history::Vector{Vector{Int}}, initial_syn::Vector{Int}, g)
    R = g.R
    # detectors
    prev = initial_syn
    lit  = Int[]
    for r in 1:R
        d = raw_history[r] .⊻ prev
        for (i, b) in enumerate(d)
            b == 1 && push!(lit, g.node_id(i, r))
        end
        prev = raw_history[r]
    end
    pairs, cost = mwpm(lit, g.dist, g.BND)
    x_qubits = Tuple{Int,Int}[]; meas_err = Tuple{Int,Int}[]
    for (a, b) in pairs
        p = g.path(a, b)
        for k in 1:(length(p)-1)
            kind = g.edge_kind(p[k], p[k+1])
            if kind[1] === :time
                push!(meas_err, kind[2])
            else
                push!(x_qubits, kind[2])
            end
        end
    end
    counts = Dict{Tuple{Int,Int}, Int}()
    for q in x_qubits; counts[q] = get(counts, q, 0) + 1; end
    frame = [q for (q, c) in counts if isodd(c)]
    (; lit, pairs, frame, meas_err, cost)
end;

## 9 · Final destructive measurement and logical $Z_L$

Logical operators in our staggered layout (X-boundaries top/bottom,
Z-boundaries left/right): $Z_L$ on each patch is the row-$y{=}0$ chain of
$Z$ operators. Combine the raw parity along that row with the parity of the
patch's accumulated Pauli frame on that row to get the logical bit.

In [9]:
const ZL_support = [(x, 0) for x in 0:(d-1)]

function measure_data_Z(psi_in; cutoff = threshold)
    psi = copy(psi_in)
    bits = Dict{Tuple{Int, Tuple{Int,Int}}, Int}()
    for p in 1:2, q in data_coords
        site = site_of[(p, q)]
        sz = real(inner(psi', apply(op("Sz", site), psi; cutoff)))
        if rand() < 0.5 + sz
            psi = apply(P_up(site), psi; cutoff); bits[(p, q)] = 0
        else
            psi = apply(P_dn(site), psi; cutoff); bits[(p, q)] = 1
        end
        psi = psi / sqrt(real(inner(psi, psi)))
    end
    bits
end

function logical_Z(bits, frame::Vector{Tuple{Int,Int}}, p::Int)
    raw          = mod(sum(bits[(p, q)]            for q in ZL_support), 2)
    frame_parity = mod(count(q -> q in ZL_support, frame), 2)
    raw ⊻ frame_parity
end;

## 10 · Full experiment

The end-to-end pipeline. One run consists of:

1. **Initialise** $|0\rangle^{n} \otimes |0\rangle^{n}$ at the data level.
2. **Codespace projection** — measure Z- and X-stabilizers once on each
   patch. Z gives all-0 (deterministic for $|0\rangle^{n}$); X gives a random
   pattern that *defines* which $|0\rangle_L$ representative we've landed in.
3. **Initial Z-syndrome readout** — $R_{\text{pre}}$ rounds of Z-stab
   measurements on both patches, decoder catches any pre-$H$ X errors.
4. **Logical $H$** on patch 1.
5. **Transversal CNOT** (patch 1 → patch 2).
6. **Final Z-syndrome readout** — $R_{\text{post}}$ rounds of Z-stab
   measurements, decoder catches any X errors that appeared during/after the
   logical operations.
7. **Destructive Z measurement** on every data qubit.
8. Combine raw $Z_L$ parity per patch with that patch's accumulated Pauli
   frame to get the logical readouts $(z_1, z_2)$.

Error injection is exposed via `physical_X_pre/post` and `meas_err_pre/post`,
which target the pre- and post-operation segments respectively.

In [10]:
# One full run of the protocol. Returns (z1, z2) plus diagnostic info.
function run_experiment(; R_pre = 2, R_post = 2,
        physical_X_pre  = Dict{Int, Vector{Tuple{Int, Tuple{Int,Int}}}}(),
        physical_X_post = Dict{Int, Vector{Tuple{Int, Tuple{Int,Int}}}}(),
        meas_err_pre    = Dict{Int, Vector{Tuple{Int, Int}}}(),
        meas_err_post   = Dict{Int, Vector{Tuple{Int, Int}}}(),
        cutoff = threshold, verbose = false)
    # 1. Initialise data |0> everywhere
    psi = MPS(sites, "Up")

    # 2. Codespace projection
    init_p1, psi = project_to_codespace!(psi, 1; cutoff)
    init_p2, psi = project_to_codespace!(psi, 2; cutoff)
    if verbose
        println("Z₀(p1)=", init_p1.z_syn,  "  X₀(p1)=", init_p1.x_syn)
        println("Z₀(p2)=", init_p2.z_syn,  "  X₀(p2)=", init_p2.x_syn)
        println("post-projection max bond dim: ", maximum(linkdims(psi)))
    end

    # 3. Pre-op Z syndrome readout
    raw_pre, psi = run_Z_rounds!(psi, R_pre;
        physical_X = physical_X_pre, meas_err = meas_err_pre, cutoff)

    # 4 + 5. Logical H and CNOT
    psi = logical_H!(psi, 1; cutoff)
    psi = logical_CNOT!(psi, 1, 2; cutoff)
    if verbose
        println("post-(H,CNOT) max bond dim: ", maximum(linkdims(psi)))
    end

    # 6. Post-op Z syndrome readout
    raw_post, psi = run_Z_rounds!(psi, R_post;
        physical_X = physical_X_post, meas_err = meas_err_post, cutoff)

    # 7. Decode each segment per patch
    g_pre  = build_graph(R_pre)
    g_post = build_graph(R_post)
    dec_pre  = (decode_patch(raw_pre[1],  init_p1.z_syn, g_pre),
                decode_patch(raw_pre[2],  init_p2.z_syn, g_pre))
    # For the post segment the "round-0 reference" is the LAST pre-segment
    # raw syndrome. (After the logical operations the actual stabilizer
    # parities can differ; for patch 1 in particular, the Z-stab readings
    # after the H+CNOT no longer match the pre-H ones. We treat the first
    # post-op round as the new reference for that patch.)
    ref_post_p1 = raw_post[1][1]    # first post-op round IS the reference
    ref_post_p2 = raw_post[2][1]    # same for patch 2 (CNOT propagation)
    dec_post = (decode_patch(raw_post[1][2:end], ref_post_p1, build_graph(R_post-1)),
                decode_patch(raw_post[2][2:end], ref_post_p2, build_graph(R_post-1)))

    # Cumulative Pauli frame per patch
    frame_p1 = vcat(dec_pre[1].frame, dec_post[1].frame)
    frame_p2 = vcat(dec_pre[2].frame, dec_post[2].frame)

    # 8. Destructive Z + logical Z_L
    bits = measure_data_Z(psi; cutoff)
    z1 = logical_Z(bits, frame_p1, 1)
    z2 = logical_Z(bits, frame_p2, 2)

    (; z1, z2, bits, frame_p1, frame_p2,
       dec_pre, dec_post, init_p1, init_p2,
       raw_pre, raw_post)
end;

## 11 · Demo A: no errors → Bell-state correlation

Run the protocol many times with no errors injected and tally the
$(z_1, z_2)$ outcomes. The state at step 7 is the logical Bell state
$|\Phi^+\rangle_L = (|00\rangle_L + |11\rangle_L)/\sqrt 2$, so we expect:

- $z_1 = z_2$ on every shot (perfect correlation).
- Within those, $\sim 50\%$ are $(0,0)$ and $\sim 50\%$ are $(1,1)$.

The $X_L$ syndromes of patch 2 (and the post-H/CNOT pattern) are random
per shot — that's where the per-trial variability comes from.

In [11]:
const NTRIALS = 30
let
    counts = Dict((0,0)=>0, (0,1)=>0, (1,0)=>0, (1,1)=>0)
    for t in 1:NTRIALS
        out = run_experiment()
        counts[(out.z1, out.z2)] += 1
    end
    println("outcome counts over $NTRIALS no-error trials:")
    for k in sort(collect(keys(counts))); println("  ", k, " → ", counts[k]); end
    correlated = counts[(0,0)] + counts[(1,1)]
    println("Z_L1 == Z_L2 in ", correlated, "/", NTRIALS, " trials",
            "  (expect ", NTRIALS, ")")
end

┌ Warning: Calling `inner(x::MPS, A::MPO, y::MPS)` where the site indices of the `MPS`
│ `x` and the `MPS` resulting from contracting `MPO` `A` with `MPS` `y` don't
│ match is deprecated as of ITensors v0.3 and will result in an error in ITensors
│ v0.4. The most common cause of this is something like the following:
│ 
│ ```julia
│ s = siteinds("S=1/2")
│ psi = random_mps(s)
│ H = MPO(s, "Id")
│ inner(psi, H, psi)
│ ```
│ 
│ `psi` has the Index structure `-s-(psi)` and `H` has the Index structure
│ `-s'-(H)-s-`, so the Index structure of would be `(dag(psi)-s- -s'-(H)-s-(psi)`
│  unless the prime levels were fixed. Previously we tried fixing the prime level
│   in situations like this, but we will no longer be doing that going forward.
│ 
│ There are a few ways to fix this. You can simply change:
│ 
│ ```julia
│ inner(psi, H, psi)
│ ```
│ 
│ to:
│ 
│ ```julia
│ inner(psi', H, psi)
│ ```
│ 
│ in which case the Index structure will be `(dag(psi)-s'-(H)-s-(psi)`.
│ 
│ Alternatively, you c

outcome counts over 30 no-error trials:
  (0, 0) → 14
  (0, 1) → 0
  (1, 0) → 0
  (1, 1) → 16
Z_L1 == Z_L2 in 30/30 trials  (expect 30)


## 12 · Demo B: single physical X error on patch 1, decoder rescues

Inject one X on patch 1's data qubit $(1, 1)$ between pre-op rounds 1 and 2.
This is the simplest correctable case — the X excites the two Z-stabs whose
support contains $(1, 1)$, so two adjacent defects light up in the
pre-segment matching graph. MWPM matches them via the edge labelled by
$(1, 1)$, and the Pauli frame for patch 1 ends up with $(1,1)$ in it.

Crucially, $(1, 1) \notin Z_L^{(1)}$ support (which is row $y=0$), so the
frame parity on $Z_L$ is 0 and the logical readout matches the
uncorrected Bell value. We should still see $Z_L^{(1)} = Z_L^{(2)}$ on
every shot.

In [11]:
let
    counts = Dict((0,0)=>0, (0,1)=>0, (1,0)=>0, (1,1)=>0)
    frames_seen = Set{Vector{Tuple{Int,Int}}}()
    for _ in 1:20
        out = run_experiment(; physical_X_pre = Dict(2 => [(1, (1, 1))]))
        counts[(out.z1, out.z2)] += 1
        push!(frames_seen, sort(out.frame_p1))
    end
    println("outcomes:")
    for k in sort(collect(keys(counts))); println("  ", k, " → ", counts[k]); end
    println("distinct patch-1 frames seen: ", frames_seen)
    println("correlated trials: ", counts[(0,0)] + counts[(1,1)], "/20")
end

┌ Warning: Calling `inner(x::MPS, A::MPO, y::MPS)` where the site indices of the `MPS`
│ `x` and the `MPS` resulting from contracting `MPO` `A` with `MPS` `y` don't
│ match is deprecated as of ITensors v0.3 and will result in an error in ITensors
│ v0.4. The most common cause of this is something like the following:
│ 
│ ```julia
│ s = siteinds("S=1/2")
│ psi = random_mps(s)
│ H = MPO(s, "Id")
│ inner(psi, H, psi)
│ ```
│ 
│ `psi` has the Index structure `-s-(psi)` and `H` has the Index structure
│ `-s'-(H)-s-`, so the Index structure of would be `(dag(psi)-s- -s'-(H)-s-(psi)`
│  unless the prime levels were fixed. Previously we tried fixing the prime level
│   in situations like this, but we will no longer be doing that going forward.
│ 
│ There are a few ways to fix this. You can simply change:
│ 
│ ```julia
│ inner(psi, H, psi)
│ ```
│ 
│ to:
│ 
│ ```julia
│ inner(psi', H, psi)
│ ```
│ 
│ in which case the Index structure will be `(dag(psi)-s'-(H)-s-(psi)`.
│ 
│ Alternatively, you c

outcomes:
  (0, 0) → 11
  (0, 1) → 0
  (1, 0) → 0
  (1, 1) → 9
distinct patch-1 frames seen: Set([[(1, 1)]])
correlated trials: 20/20


## 13 · Demo C: $X$ on patch 2 post-CNOT, decoder rescues

Inject one X on patch 2's data $(1, 1)$ between post-CNOT rounds 1 and 2.
Patch 2's post-segment decoder should see two defects there at round 2 (the
syndrome change between round 1 and round 2), match them, and add $(1, 1)$
to patch 2's frame. Correlation preserved.

In [12]:
let
    counts = Dict((0,0)=>0, (0,1)=>0, (1,0)=>0, (1,1)=>0)
    for _ in 1:20
        out = run_experiment(; physical_X_post = Dict(2 => [(2, (1, 1))]))
        counts[(out.z1, out.z2)] += 1
    end
    println("outcomes:")
    for k in sort(collect(keys(counts))); println("  ", k, " → ", counts[k]); end
    println("correlated trials: ", counts[(0,0)] + counts[(1,1)], "/20")
end

outcomes:
  (0, 0) → 10
  (0, 1) → 0
  (1, 0) → 0
  (1, 1) → 10
correlated trials: 20/20


## 14 · Demo D: undetected logical $X_L$ on **patch 2** → anti-correlated

This is the failure mode. Inject the full $X_L^{(2)}$ chain
($X$ on every data qubit in column 0 of patch 2) before round 1 of the
pre-segment. $X_L^{(2)}$ commutes with every Z-stabilizer of patch 2, so
patch 2's decoder sees no defects.

Why patch 2 specifically: a transversal CNOT with patch 1 as control
propagates X from patch 1 to patch 2 (and not the reverse), so an
$X_L^{(1)}$ injected pre-CNOT becomes $X_L^{(1)} X_L^{(2)}$ post-CNOT, which
*commutes* with $Z_L^{(1)} Z_L^{(2)}$ and so still keeps the Bell
correlation. An $X_L^{(2)}$ does not propagate back to patch 1, so it
remains $X_L^{(2)}$, which **anti-commutes** with $Z_L^{(1)} Z_L^{(2)}$ and
flips the correlation: outcomes become $(0, 1)$ and $(1, 0)$ instead of
$(0, 0)$ and $(1, 1)$.

If you re-run with the chain on patch 1 instead, you'll see the correlation
restored — that's the asymmetry the CNOT direction introduces.

In [ ]:
let
    XL2 = [(2, (0, y)) for y in 0:(d-1)]   # full X_L chain on patch 2
    counts = Dict((0,0)=>0, (0,1)=>0, (1,0)=>0, (1,1)=>0)
    for _ in 1:20
        out = run_experiment(; physical_X_pre = Dict(1 => XL2))
        counts[(out.z1, out.z2)] += 1
    end
    println("outcomes (X_L on patch 2 before CNOT):")
    for k in sort(collect(keys(counts))); println("  ", k, " → ", counts[k]); end
    println("anti-correlated trials: ", counts[(0,1)] + counts[(1,0)], "/20  (expect ~20)")
end

## 15 · Demo E: measurement error during a syndrome round

Inject a classical bit-flip on Z-aux index 1 of patch 1 in round 1 of the
pre-segment (the recorded bit is wrong; the underlying qubit state is fine).
The reset-model time edges $r \leftrightarrow r{+}1$ should pair it cleanly
between rounds 1 and 2 of the pre-segment, so the decoder identifies the
measurement error and *does not* add anything to the Pauli frame.

(With only two pre-segment rounds this is borderline: the first
post-correction round is the reference for the post-segment, so any
mis-decoded measurement error here would show up as a frame entry. The
demo's check is that no spurious frame entries appear on patch 1.)

In [ ]:
let
    out = run_experiment(; meas_err_pre = Dict(1 => [(1, 1)]), verbose = true)
    println()
    println("(z1, z2)         = ", (out.z1, out.z2))
    println("frame patch 1    = ", out.frame_p1)
    println("frame patch 2    = ", out.frame_p2)
    println("decoder pre.p1   = (lit=", out.dec_pre[1].lit,
            ", meas_err=", out.dec_pre[1].meas_err, ")")
end

## Caveats / next steps

- **$d=3$ for tractability.** Crank up to $d=5$ once you're satisfied — that
  costs 49-site non-local CNOTs and runtimes minutes per trial in the
  current setup. Hitting $d=7$ or two qubits beyond $d=5$ will probably
  require either an interleaved-by-position ordering (sacrificing the
  scaling-honesty of side-by-side) or a tensor-network format that handles
  2-D entanglement natively.
- **Post-op decoder reference is the first post-op round.** Errors that
  appear *immediately* after the H+CNOT (before any post-op syndrome round)
  are missed — they look like part of the reference. A cleaner protocol
  would track the expected syndrome shift from H and CNOT analytically and
  use that as the reference, or run an extra noise-free round just after
  the operations to lock in a known reference.
- **Only X errors are decoded.** Z errors on the data are invisible to
  Z-stabilizers but harmless to $|0\rangle_L$ (which is a $Z_L$ eigenstate)
  — so for a Z-basis readout we don't need to decode them. For an X-basis
  readout, or anything beyond pure Bell-state preparation, you'd want
  X-stab measurements run round-by-round too and a parallel Z-error
  decoder.
- **Side-by-side MPS layout** is the scaling-honest choice: every
  inter-patch operation pays the non-locality cost an actual hardware
  simulation would face. Interleaving would hide that and make scaling
  predictions optimistic.